In [1]:
import numpy as np
import pandas as pd 

## pipeline 
- Start with the Raw Files: Your cvedataset-patches folder is your primary source.

- Validate with Metadata: Use the CVEfixes_v1.0.8.db to get the definitive mapping between   cve_id, commit_hash, and repo. Look for tables like cves and commits.

- Parse and Clean: Use a multi-language crawler library like unidiff to parse the diffs, and then clean the data by filtering file extensions and removing empty entries.

- Create Your Index: Once you have your cleaned vuln_code and patched_code pairs, you can feed vuln_code into your embedding pipeline and build your FAISS index for retrieval.
```
🧠 Goal

Convert messy patches → clean dataset:

{
  "vulnerable_code": "...",
  "fixed_code": "...",
  "language": "...",
  "file": "...",
  "cve": "..."
}
```



## Process Entire Dataset

In [8]:
import os
import re
import json

# -----------------------------
# 🔹 Language Detection
# -----------------------------
def detect_language(filename):
    if not filename:
        return "unknown"

    ext = filename.split('.')[-1].lower()

    mapping = {
        "py": "python",
        "js": "javascript",
        "ts": "typescript",
        "java": "java",
        "cpp": "cpp",
        "c": "c",
        "cs": "csharp",
        "go": "go",
        "rb": "ruby",
        "php": "php",
        "html": "html",
        "css": "css"
    }
    return mapping.get(ext, "unknown")


# -----------------------------
# 🔹 Pattern Detection (Basic)
# -----------------------------
def detect_pattern(code):
    code = code.lower()

    if "select" in code and "+" in code:
        return "sql_injection"
    if "exec(" in code or "system(" in code:
        return "command_injection"
    if "html" in code or "<script>" in code:
        return "xss"
    if "password" in code:
        return "auth"
    return "general"


# -----------------------------
# 🔹 Validation Filter
# -----------------------------
def is_valid_sample(vuln, fix):
    if not vuln or not fix:
        return False
    if vuln.strip() == fix.strip():
        return False
    if len(vuln.split()) < 3:
        return False
    if len(fix.split()) < 3:
        return False
    return True


# -----------------------------
# 🔹 Patch Parser (ROBUST)
# -----------------------------
def parse_patch_file(filepath):
    samples = []

    vulnerable_lines = []
    fixed_lines = []
    current_file = "unknown"

    try:
        with open(filepath, "r", errors="ignore") as f:
            for line in f:
                line = line.lstrip()

                # detect file name if exists
                if "diff --git" in line:
                    match = re.search(r"b/(.*)", line)
                    if match:
                        current_file = match.group(1)

                # skip metadata
                if line.startswith(("---", "+++", "@@")):
                    continue

                # vulnerable lines
                if line.startswith("-") and not line.startswith("---"):
                    vulnerable_lines.append(line[1:].strip())

                # fixed lines
                elif line.startswith("+") and not line.startswith("+++"):
                    fixed_lines.append(line[1:].strip())

        # save sample if valid
        if vulnerable_lines and fixed_lines:
            samples.append({
                "file": current_file,
                "vulnerable": "\n".join(vulnerable_lines),
                "fixed": "\n".join(fixed_lines)
            })

    except Exception as e:
        print(f"Error reading {filepath}: {e}")

    return samples


# -----------------------------
# 🔹 Main Processing Pipeline
# -----------------------------
def process_dataset(root_dir):
    dataset = []
    total_files = 0
    total_samples = 0

    for root, _, files in os.walk(root_dir):
        for file in files:
            total_files += 1

            path = os.path.join(root, file)

            samples = parse_patch_file(path)

            if samples:
                print(f"[✔] Found samples in: {path}")

            for s in samples:
                vuln = s["vulnerable"]
                fix = s["fixed"]

                if not is_valid_sample(vuln, fix):
                    continue

                item = {
                    "vulnerable_code": vuln,
                    "fixed_code": fix,
                    "file": s["file"],
                    "language": detect_language(s["file"]),
                    "pattern": detect_pattern(vuln)
                }

                dataset.append(item)
                total_samples += 1

    print("\n📊 SUMMARY")
    print("Total files scanned:", total_files)
    print("Total valid samples:", total_samples)

    return dataset


# -----------------------------
# 🔹 SAVE DATASET
# -----------------------------
def save_dataset(dataset, output_file="final_dataset.jsonl"):
    with open(output_file, "w") as f:
        for item in dataset:
            f.write(json.dumps(item) + "\n")

    print(f"\n💾 Dataset saved to {output_file}")


# -----------------------------
# 🔹 RUN EVERYTHING
# -----------------------------
if __name__ == "__main__":
    ROOT_DIR = "../dataset/cvedataset-patches"  # 🔥 CHANGE if needed

    data = process_dataset(ROOT_DIR)

    # print sample output
    print("\n🔍 SAMPLE OUTPUT:")
    for i in range(min(5, len(data))):
        print(json.dumps(data[i], indent=2))
        print("-" * 50)

    save_dataset(data)

[✔] Found samples in: ../dataset/cvedataset-patches/github.com_snapcore_snapcraft_a0ceca9d531a34c979251030ed67b5fa2abfdd9a.patch
[✔] Found samples in: ../dataset/cvedataset-patches/github.com_froxlor_froxlor_94d9c3eedf31bc8447e3aa349e32880dde02ee52.patch
[✔] Found samples in: ../dataset/cvedataset-patches/github.com_netwide-assembler_nasm_f0ceb1e122dc3523123dd8dfd6113f2e68451452.patch
[✔] Found samples in: ../dataset/cvedataset-patches/github.com_torvalds_linux_c96988b7d99327bb08bd9efd29a203b22cd88ace.patch
[✔] Found samples in: ../dataset/cvedataset-patches/github.com_eclipse-theia_theia_0761dcf5fe3c14c27432683d42d2c526ad0cfbd5.patch
[✔] Found samples in: ../dataset/cvedataset-patches/github.com_julianlam_nodebb-plugin-markdown_ab7f2684750882f7baefbfa31db8d5aac71e6ec3.patch
[✔] Found samples in: ../dataset/cvedataset-patches/github.com_onelogin_ruby-saml_048a544730930f86e46804387a6b6fad50d8176f.patch
[✔] Found samples in: ../dataset/cvedataset-patches/github.com_knik0_faad2_942c3e0aee

In [2]:
df = pd.read_json("final_dataset.jsonl",lines=True)

In [4]:
df.head()

,vulnerable_code,fixed_code,file,language,pattern
0,project_loader: do not export empty environmen...,"combined_paths = combine_paths(paths, prepend,...",tests/unit/test_formatting_utils.py,python,general
1,"__drm_atomic_helper_crtc_reset(crtc, &cstate->...",if (cstate)\n__drm_atomic_helper_crtc_reset(cr...,drivers/gpu/drm/msm/disp/dpu1/dpu_crtc.c,c,general
2,"""THEIA_WEBVIEW_EXTERNAL_ENDPOINT"": ""${env:THEI...","""THEIA_WEBVIEW_EXTERNAL_ENDPOINT"": ""${env:THEI...",packages/plugin-ext/src/main/node/plugin-servi...,typescript,general
3,"var\tRemarkable = require('remarkable'),\npars...","var\tMarkdownIt = require('markdown-it'),\npar...",package.json,unknown,general
4,node.text if node\ncerts['signing'] << cert_no...,Utils.element_text(node)\ncerts['signing'] << ...,test/responses/response_node_text_attack.xml.b...,unknown,general


In [3]:
df.isna().sum()

vulnerable_code    0
fixed_code         0
file               0
language           0
pattern            0
dtype: int64

## creating temp embeddings -- just to see the dataset

In [5]:
import sklearn

In [5]:
# ! pip install matplotlib


In [6]:
import json

data = []
with open("final_dataset.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

print("Dataset size:", len(data))

Dataset size: 25321


In [7]:
texts = [
    d["vulnerable_code"] + "\n" + d["fixed_code"]
    for d in data[:2000]
]

print("Texts created:", len(texts))
print(texts[0])

Texts created: 2000
project_loader: do not export empty environment
meta: do not export empty environment. Warn on empty environment.

return '{envvar}="${envvar}{separator}{paths}"'.format(
envvar=envvar,
separator=separator,
paths=combine_paths(paths, prepend, separator),
)
env.append('export LD_LIBRARY_PATH="$SNAP_LIBRARY_PATH:$LD_LIBRARY_PATH"')
'LD_LIBRARY_PATH="' + ":".join(dependency_paths) + ':$LD_LIBRARY_PATH"'
+ ":".join(
["{0}/usr/sbin", "{0}/usr/bin", "{0}/sbin", "{0}/bin", "$PATH"]
).format(root)
+ '"'
export PATH="$SNAP/usr/sbin:$SNAP/usr/bin:$SNAP/sbin:$SNAP/bin:$PATH"
export LD_LIBRARY_PATH="$SNAP_LIBRARY_PATH:$LD_LIBRARY_PATH"
export LD_LIBRARY_PATH="$SNAP_LIBRARY_PATH:$LD_LIBRARY_PATH"
export PATH="$SNAP/usr/sbin:$SNAP/usr/bin:$SNAP/sbin:$SNAP/bin:$PATH"
export LD_LIBRARY_PATH="$SNAP_LIBRARY_PATH:$LD_LIBRARY_PATH"
export PATH="$SNAP/usr/sbin:$SNAP/usr/bin:$SNAP/sbin:$SNAP/bin:$PATH"
export LD_LIBRARY_PATH="$SNAP_LIBRARY_PATH:$LD_LIBRARY_PATH"
export PATH="$SNAP/usr/sb

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-large-en")

embeddings = model.encode(
    texts,   # 🔥 THIS WAS MISSING
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Shape:", embeddings.shape)

/home/muhammad-taaha/code/repo-llm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-large-en")

# ⚠️ Always limit samples (VERY IMPORTANT)
X = np.array(embeddings[:2000])

# ✅ t-SNE works better after normalization
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

# ✅ Run t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    n_iter=1000,
    random_state=42,
    init="pca"
)

reduced = tsne.fit_transform(X)

# ✅ Plot
plt.figure(figsize=(8, 6))
plt.scatter(reduced[:, 0], reduced[:, 1], s=5)
plt.title("t-SNE Visualization of Code Embeddings")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.show()

/home/muhammad-taaha/code/repo-llm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'embeddings' is not defined

## i dont have the local gpus so the training is not possible i will use kaggle to make the embeddings